In [0]:
from pyspark import SparkFiles
Churn_Modelling ="https://github.com/erkansirin78/datasets/raw/master/Churn_Modelling.csv"
sc.addFile(Churn_Modelling)
df = spark.read.csv("file:///"+SparkFiles.get("Churn_Modelling.csv"), header= True, inferSchema=True)

In [0]:
display(df.limit(5))

RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
1,15634602,Hargrave,619,France,Female,42,2,0.0,1,1,1,101348.88,1
2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
3,15619304,Onio,502,France,Female,42,8,159660.8,3,1,0,113931.57,1
4,15701354,Boni,699,France,Female,39,1,0.0,2,0,0,93826.63,0
5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0


In [0]:
categoric_cols = []
numeric_cols = []
discarted_cols = ['RowNumber','CustomerId','Surname']
label_col = ['Exited']

In [0]:
# column seperation
for col_name in df.dtypes:
    if (col_name[0] not in label_col+discarted_cols):
        if (col_name[1] == 'string'):
            categoric_cols.append(col_name[0])
        else: numeric_cols.append(col_name[0])

In [0]:
# Here we count the distinct categories in categoric columns. If there is more than two,
# we will add those to to_be_onehotencoded_cols list If there is just 2 we don't need to use onehotencoder
# So if distinct category gt 2 we have to add it to_be_onehotencoded_cols list
to_be_onehotencoded_cols = []

for col_name in categoric_cols:
    count = df.select(col_name).distinct().count()
    if count > 2:
        to_be_onehotencoded_cols.append(col_name)
    print("{} has {} distinct category.".format(col_name, count))

Geography has 3 distinct category.
Gender has 2 distinct category.


In [0]:
from pyspark.ml.feature import StringIndexer

# Will hold stringIndexer objects and column names
my_dict = {}

# Will collect StringIndexer ojects
string_indexer_objs = []

# Will collect StringIndexer output colnames
string_indexer_output_names = []

# Will collect OneHotEncoder output colnames
ohe_col_input_names = []
ohe_col_output_names = []

for col_name in categoric_cols:
    my_dict[col_name+"_indexobj"] = StringIndexer() \
    .setHandleInvalid('skip') \
    .setInputCol(col_name) \
    .setOutputCol(col_name+"_indexed")
    
    string_indexer_objs.append(my_dict.get(col_name+"_indexobj"))
    string_indexer_output_names.append(col_name+"_indexed")
    if col_name in to_be_onehotencoded_cols:
        ohe_col_input_names.append(col_name+"_indexed")
        ohe_col_output_names.append(col_name+"_ohe")

In [0]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder() \
.setInputCols(ohe_col_input_names) \
.setOutputCols(ohe_col_output_names)

In [0]:
from pyspark.ml.feature import VectorAssembler

string_indexer_col_names_ohe_exluded = list(set(string_indexer_output_names).difference(set(ohe_col_input_names)))

assembler = VectorAssembler().setHandleInvalid("skip") \
.setInputCols(numeric_cols+string_indexer_col_names_ohe_exluded+ohe_col_output_names) \
.setOutputCol('unscaled_features')

In [0]:
from pyspark.ml.feature import StandardScaler
scaler = StandardScaler().setInputCol("unscaled_features").setOutputCol("features")

In [0]:
from pyspark.ml.clustering import KMeans

kmeansObject = KMeans()

In [0]:
from pyspark.ml import Pipeline

pipeline_obj = Pipeline().setStages(string_indexer_objs+[encoder, assembler, scaler, kmeansObject])

In [0]:
pipeline_model = pipeline_obj.fit(df)

In [0]:
transformed_df = pipeline_model.transform(df)

In [0]:
display(transformed_df.limit(5))

RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_indexed,Gender_indexed,Geography_ohe,unscaled_features,features,prediction
1,15634602,Hargrave,619,France,Female,42,2,0.0,1,1,1,101348.88,1,0.0,1.0,"Map(vectorType -> sparse, length -> 2, indices -> List(0), values -> List(1.0))","Map(vectorType -> dense, length -> 11, values -> List(619.0, 42.0, 2.0, 0.0, 1.0, 1.0, 1.0, 101348.88, 1.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 11, values -> List(6.404333924389993, 4.0046505619078925, 0.691521236019042, 0.0, 1.7192340885061048, 2.1937499584452778, 2.0008126162027735, 1.7622676321217514, 2.0083062457076424, 1.9999078371539627, 0.0))",1
2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2.0,1.0,"Map(vectorType -> sparse, length -> 2, indices -> List(), values -> List())","Map(vectorType -> dense, length -> 11, values -> List(608.0, 41.0, 1.0, 83807.86, 1.0, 0.0, 1.0, 112542.58, 1.0, 0.0, 0.0))","Map(vectorType -> dense, length -> 11, values -> List(6.2905250824379895, 3.9093017390053237, 0.345760618009521, 1.3431305312804176, 1.7192340885061048, 0.0, 2.0008126162027735, 1.9569051574074896, 2.0083062457076424, 0.0, 0.0))",0
3,15619304,Onio,502,France,Female,42,8,159660.8,3,1,0,113931.57,1,0.0,1.0,"Map(vectorType -> sparse, length -> 2, indices -> List(0), values -> List(1.0))","Map(vectorType -> dense, length -> 11, values -> List(502.0, 42.0, 8.0, 159660.8, 3.0, 1.0, 0.0, 113931.57, 1.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 11, values -> List(5.193821696355051, 4.0046505619078925, 2.766084944076168, 2.5587730688822803, 5.1577022655183145, 2.1937499584452778, 0.0, 1.9810570978960358, 2.0083062457076424, 1.9999078371539627, 0.0))",1
4,15701354,Boni,699,France,Female,39,1,0.0,2,0,0,93826.63,0,0.0,1.0,"Map(vectorType -> sparse, length -> 2, indices -> List(0), values -> List(1.0))","Map(vectorType -> dense, length -> 11, values -> List(699.0, 39.0, 1.0, 0.0, 2.0, 0.0, 0.0, 93826.63, 1.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 11, values -> List(7.232034593131834, 3.718604093200186, 0.345760618009521, 0.0, 3.4384681770122096, 0.0, 0.0, 1.631469761481959, 2.0083062457076424, 1.9999078371539627, 0.0))",1
5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0,2.0,1.0,"Map(vectorType -> sparse, length -> 2, indices -> List(), values -> List())","Map(vectorType -> dense, length -> 11, values -> List(850.0, 43.0, 2.0, 125510.82, 1.0, 1.0, 1.0, 79084.1, 1.0, 0.0, 0.0))","Map(vectorType -> dense, length -> 11, values -> List(8.794319605382057, 4.099999384810461, 0.691521236019042, 2.0114749899119353, 1.7192340885061048, 2.1937499584452778, 2.0008126162027735, 1.3751247142097653, 2.0083062457076424, 0.0, 0.0))",0


In [0]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator() \
      .setFeaturesCol("features") \
      .setPredictionCol("prediction") \
      .setMetricName("silhouette")

In [0]:
evaluator.evaluate(transformed_df)

Out[16]: 0.24590625242752784

In [0]:
from pyspark.ml.tuning import CrossValidatorModel, ParamGridBuilder, CrossValidator

paramGrid = ParamGridBuilder() \
.addGrid(kmeansObject.k, [3,5,7,9,11]) \
.build()

In [0]:
crossval = CrossValidator(estimator=pipeline_obj,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3) 

In [0]:
import time
start_time = time.time()
cvModel = crossval.fit(df)

print("----------- %s secs ----------" %(time.time() - start_time))

In [0]:
cvModel.avgMetrics

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-2966764295380429>:1
----> 1 cvModel.avgMetrics

NameError: name 'cvModel' is not defined

In [0]:
cvModel.write().overwrite().save("file:///saved_models/kmeans_cvmodel")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-2966764295380430>:1
----> 1 cvModel.write().overwrite().save("file:///saved_models/kmeans_cvmodel")

NameError: name 'cvModel' is not defined

In [0]:
loaded_model = CrossValidatorModel.load("file:///saved_models/kmeans_cvmodel")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-2966764295380431>:1
----> 1 loaded_model = CrossValidatorModel.load("file:///saved_models/kmeans_cvmodel")

File /databricks/spark/python/pyspark/ml/util.py:447, in MLReadable.load(cls, path)
    444 @classmethod
    445 def load(cls, path: str) -> RL:
    446     """Reads an ML instance from the input path, a shortcut of `read().load(path)`."""
--> 447     return cls.read().load(path)

File /databricks/spark/python/pyspark/ml/tuning.py:545, in CrossValidatorModelReader.load(self, path)
    544 def load(self, path: str) -> "CrossValidatorModel":
--> 545     metadata = DefaultParamsReader.loadMetadata(path, self.sc)
    546     if not DefaultParamsReader.isPythonParamsInstance(metadata):
    547         return JavaMLReader(self.cls).load(path)  # type: ignore[arg-type]

File /databricks/spark/python/pyspark/ml/util.py:659, 

In [0]:
loaded_model.transform(df.limit(5)).toPandas()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-2966764295380432>:1
----> 1 display(loaded_model.transform(df.limit(5)))

NameError: name 'loaded_model' is not defined

In [0]:
loaded_model.transform(df).groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|         1| 3566|
|         2| 2509|
|         0| 3925|
+----------+-----+



In [0]:
pipelineModel = loaded_model.getEstimator()

In [0]:
type(pipelineModel)

Out[41]: pyspark.ml.pipeline.Pipeline

In [0]:
type(pipelineModel.getStages()[-1])

Out[47]: pyspark.ml.clustering.KMeans

In [0]:
# What is the k value?
pipelineModel.getStages()[-1].getK()

Out[54]: 2

In [0]:
score = evaluator.evaluate(loaded_model.transform(df))

score

Out[55]: 0.22645984369837002